# Data Reading

### Reading CSV Files 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = (spark
      .read
      .format("csv")
      .option("header",True)
      .option("inferSchema",True)
      .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv"))
df.display()

### Reading JSON Files 

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

### Reading Parquet

In [0]:
df_parquet = (spark.
              read
              .format("parquet")
              .load("/Volumes/learnspark/raw/spark_volume/raw_orders/part-00000-tid-orders.c000.snappy.parquet"))
df_parquet.display()

### Reading JDBC

In [0]:
# my_url = "jdbc:postgresql://localhost:5432/postgres"
# myconnection = {"user": "postgres", 
#                 "password": "postgres",
#                 "driver": "org.postgresql.Driver"}
# df = (spark
#       .read
#       .jdbc(url = my_url, table = "orders",properties= myconnection))

# # OR

# df = (
#     spark.read
#     .format("jdbc")
#     .option("url", my_url)
#     .option("dbtable", "orders")
#     .option("user", "postgres")
#     .option("password", "postgres")
#     .option("driver", "org.postgresql.Driver")
#     .load()
# )

# CORRUPT RECORDS MODES

### Permissive 
- reads the complete file, if any corrupt records are they it will be stored in seperate column 

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .option("mode","PREMISSIVE")
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

### DROPMALFORMED 
- it will simple drop malformed record while reading

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .option("mode","DROPMALFORMED")
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

### FAILFAST
- if your downstream application is too sensitive that you cannot take risk, then you can use mode = FAILFAST, so that pipeline will fail immediately 

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .option("mode","FAILFAST")
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json")
           )

df_json.display()

# DATA SCHEMA

### StructType Method

In [0]:
df.schema

In [0]:
my_custom_schme = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_date', DateType(), True), StructField('product_id', StringType(), True), StructField('quantity', IntegerType(), True), StructField('price', DoubleType(), True), StructField('order_status', StringType(), True), StructField('shipping_address', StringType(), True), StructField('city', StringType(), True), StructField('country', StringType(), True), StructField('payment_method', StringType(), True), StructField('discount', DoubleType(), True), StructField('category', StringType(), True), StructField('sales_rep', StringType(), True), StructField('region', StringType(), True), StructField('ship_date', DateType(), True), StructField('delivery_days', IntegerType(), True), StructField('returned', StringType(), True), StructField('gender', StringType(), True)])

In [0]:
df_csv = (spark
          .read
          .format("csv")
          .option("header",True)
          .schema(my_custom_schme)
          .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv"))

df_csv.display()

### DDL Schema

In [0]:
my_ddl_schema = """
order_id INT,
customer_id STRING,
order_date DATE,
product_id STRING,
quantity INTEGER,
price DOUBLE,
order_status STRING,
shipping_address STRING,
city STRING,
country STRING,
payment_method STRING,
discount DOUBLE,
category STRING,
sales_rep STRING,
region STRING,
ship_date DATE,
delivery_days INTEGER,
returned STRING,
gender STRING
"""
df_csv = (spark
          .read
          .format("csv")
          .option("header",True)
          .schema(my_ddl_schema)
          .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv")
         )
display(df_csv)

### SELECT

In [0]:
df_select = df_csv.select("city","country","category")
# OR
df_select = df_csv.select(col("city"),col("country"),col("category"))

df_select.display()

### Alias

In [0]:
df_alias = df_csv.select(col("city").alias("cutomer_city"),col("country").alias("customer_country"),"category")
df_alias.display()

### Filter

#### Scenario_1

In [0]:
display(df_csv.filter(
    col("order_status") == "Returned"
    ))


#### scenario_2

In [0]:
display(df_csv.filter(
    (col("order_status") == "Returned") |
    (col("order_status") == "Cancelled")
    ))

#### isin

In [0]:
desired_order = ["Returned","Cancelled","Shipped"]
display(df_csv.filter(
    col("order_status").isin("Returned","Cancelled","Shipped")
    ))
# OR
display(df_csv.filter(
    col("order_status").isin(desired_order)
    ))

#### withColumnRenamed

In [0]:
df.withColumnRenamed("order_status","status").display()

#### withColumn
- This is the go-to API for either transforming the column or adding/creating a new one

##### scenario_1

In [0]:
df.withColumn("file_path",input_file_name()).withColumn('flag',lit('0')).display()

##### scenario-2

In [0]:
display(
    df.withColumn('shipping_address',regexp_replace("shipping_address",',.*',''))
)

##### scenario-3

In [0]:
display(
    df.withColumn("Total_Price",round(col("price")*col("quantity"),2))
)
# OR
display(
    df.withColumn("Total_Price",col("price")*col("quantity")).withColumn("Total_Price",round("Total_Price",2))
)

### TypeCasting

In [0]:
display(df.withColumn("order_id",col("order_id").cast(StringType())))
# OR
display(df.withColumn("order_id",col("order_id").cast("STRING")))

### Sorting

#### scenario-1

In [0]:
df.sort(col("order_date").desc()).display()

#### scenario-2

In [0]:
display(df.sort(["order_date","quantity"],ascending=[0,1]))